In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import lightgbm as lgb

### Load Data

Loading `train.csv` and `test.csv` into pandas DataFrames.

In [ ]:
try:
    train_df = pd.read_csv('/content/train.csv')
    test_df = pd.read_csv('/content/test.csv')
    print("Train and test datasets loaded successfully.")
except FileNotFoundError:
    print("Ensure 'train.csv' and 'test.csv' are uploaded to the /content/ directory.")
    train_df = pd.DataFrame()
    test_df = pd.DataFrame()

Train and test datasets loaded successfully.


### Initial Data Inspection

Displaying the first 5 rows, information summary, and descriptive statistics for both training and testing datasets.

In [ ]:
if not train_df.empty:
    print("\n--- Train Data Head ---")
    display(train_df.head())
    print("\n--- Train Data Info ---")
    display(train_df.info())
    print("\n--- Train Data Description ---")
    display(train_df.describe())


--- Train Data Head ---


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy



--- Train Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  object 
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 6.5+ MB


None


--- Train Data Description ---


,Index,day,demand,NumberofLanes,Temperature
count,77299.000000,77299.000000,7.729900e+04,77299.000000,74804.000000
mean,38649.000000,48.101838,9.394238e-02,2.014334,16.405354
std,22314.443566,0.302438,1.421905e-01,0.904665,7.359835
min,0.000000,48.000000,6.245650e-07,1.000000,-14.935097
25%,19324.500000,48.000000,1.822723e-02,1.000000,11.430473
50%,38649.000000,48.000000,4.775994e-02,2.000000,16.382587
75%,57973.500000,48.000000,1.085951e-01,3.000000,21.298833
max,77298.000000,49.000000,1.000000e+00,5.000000,48.251433


In [ ]:
if not test_df.empty:
    print("\n--- Test Data Head ---")
    display(test_df.head())
    print("\n--- Test Data Info ---")
    display(test_df.info())
    print("\n--- Test Data Description ---")
    display(test_df.describe())


--- Test Data Head ---


,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy



--- Test Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          41778 non-null  int64  
 1   geohash        41778 non-null  object 
 2   day            41778 non-null  int64  
 3   timestamp      41778 non-null  object 
 4   RoadType       41454 non-null  object 
 5   NumberofLanes  41778 non-null  int64  
 6   LargeVehicles  41778 non-null  object 
 7   Landmarks      41778 non-null  object 
 8   Temperature    40429 non-null  float64
 9   Weather        41347 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 3.2+ MB


None


--- Test Data Description ---


,Index,day,NumberofLanes,Temperature
count,41778.00000,41778.0,41778.000000,40429.000000
mean,20888.50000,49.0,2.068984,16.457339
std,12060.41411,0.0,0.988519,7.351067
min,0.00000,49.0,1.000000,-21.314645
25%,10444.25000,49.0,1.000000,11.527029
50%,20888.50000,49.0,2.000000,16.471232
75%,31332.75000,49.0,3.000000,21.365883
max,41777.00000,49.0,5.000000,45.230408


### Handle Missing Values

Addressing missing values in `RoadType`, `Temperature`, and `Weather` columns. For categorical columns, I'll impute with the mode, and for numerical 'Temperature', I'll use the mean.

In [ ]:
if not train_df.empty and not test_df.empty:
    # Impute missing values for categorical columns with mode
    for col in ['RoadType', 'Weather']:
        train_mode = train_df[col].mode()[0]
        test_mode = test_df[col].mode()[0]
        train_df[col].fillna(train_mode, inplace=True)
        test_df[col].fillna(test_mode, inplace=True)

    # Impute missing values for numerical 'Temperature' with mean
    train_mean_temp = train_df['Temperature'].mean()
    test_mean_temp = test_df['Temperature'].mean()
    train_df['Temperature'].fillna(train_mean_temp, inplace=True)
    test_df['Temperature'].fillna(test_mean_temp, inplace=True)

    print("Missing values handled for 'RoadType', 'Temperature', and 'Weather'.")

    print("\n--- Train Data Info after imputation ---")
    display(train_df.info())
    print("\n--- Test Data Info after imputation ---")
    display(test_df.info())

Missing values handled for 'RoadType', 'Temperature', and 'Weather'.

--- Train Data Info after imputation ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  object 
 4   demand         77299 non-null  float64
 5   RoadType       77299 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    77299 non-null  float64
 10  Weather        77299 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 6.5+ MB


/tmp/ipykernel_7315/403061078.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df[col].fillna(train_mode, inplace=True)
/tmp/ipykernel_7315/403061078.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usin

None


--- Test Data Info after imputation ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          41778 non-null  int64  
 1   geohash        41778 non-null  object 
 2   day            41778 non-null  int64  
 3   timestamp      41778 non-null  object 
 4   RoadType       41778 non-null  object 
 5   NumberofLanes  41778 non-null  int64  
 6   LargeVehicles  41778 non-null  object 
 7   Landmarks      41778 non-null  object 
 8   Temperature    41778 non-null  float64
 9   Weather        41778 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 3.2+ MB


None

### Convert Data Types

Converting the `timestamp` column to datetime objects for both train and test datasets. This will enable extraction of time-based features.

In [ ]:
if not train_df.empty and not test_df.empty:
    train_df['timestamp'] = pd.to_datetime(train_df['timestamp'], format='%H:%M')
    test_df['timestamp'] = pd.to_datetime(test_df['timestamp'], format='%H:%M')
    print("Timestamp columns converted to datetime objects.")

    print("\n--- Train Data Info after type conversion ---")
    display(train_df.info())
    print("\n--- Test Data Info after type conversion ---")
    display(test_df.info())

Timestamp columns converted to datetime objects.

--- Train Data Info after type conversion ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Index          77299 non-null  int64         
 1   geohash        77299 non-null  object        
 2   day            77299 non-null  int64         
 3   timestamp      77299 non-null  datetime64[ns]
 4   demand         77299 non-null  float64       
 5   RoadType       77299 non-null  object        
 6   NumberofLanes  77299 non-null  int64         
 7   LargeVehicles  77299 non-null  object        
 8   Landmarks      77299 non-null  object        
 9   Temperature    77299 non-null  float64       
 10  Weather        77299 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(3), object(5)
memory usage: 6.5+ MB


None


--- Test Data Info after type conversion ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Index          41778 non-null  int64         
 1   geohash        41778 non-null  object        
 2   day            41778 non-null  int64         
 3   timestamp      41778 non-null  datetime64[ns]
 4   RoadType       41778 non-null  object        
 5   NumberofLanes  41778 non-null  int64         
 6   LargeVehicles  41778 non-null  object        
 7   Landmarks      41778 non-null  object        
 8   Temperature    41778 non-null  float64       
 9   Weather        41778 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(3), object(5)
memory usage: 3.2+ MB


None

### Feature Engineering: Time-based Features

Extracting 'hour' and 'minute' from the `timestamp` column to capture time-of-day patterns.

In [ ]:
if not train_df.empty and not test_df.empty:
    train_df['hour'] = train_df['timestamp'].dt.hour
    train_df['minute'] = train_df['timestamp'].dt.minute
    test_df['hour'] = test_df['timestamp'].dt.hour
    test_df['minute'] = test_df['timestamp'].dt.minute
    print("Extracted 'hour' and 'minute' features from timestamp.")

    # Drop the original timestamp column as its information has been extracted
    train_df.drop('timestamp', axis=1, inplace=True)
    test_df.drop('timestamp', axis=1, inplace=True)
    print("Original 'timestamp' column dropped.")

    print("\n--- Train Data Head after feature engineering ---")
    display(train_df.head())
    print("\n--- Test Data Head after feature engineering ---")
    display(test_df.head())

Extracted 'hour' and 'minute' features from timestamp.
Original 'timestamp' column dropped.

--- Train Data Head after feature engineering ---


,Index,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,minute
0,0,qp02z1,48,0.048804,Residential,1,Not Allowed,No,16.405354,Sunny,0,0
1,1,qp02zt,48,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0
2,2,qp08bj,48,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0
3,3,qp08gt,48,0.003272,Residential,1,Not Allowed,No,16.405354,Rainy,0,0
4,4,qp02zq,48,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0



--- Test Data Head after feature engineering ---


,Index,geohash,day,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,minute
0,0,qp02z1,49,Residential,1,Not Allowed,No,16.457339,Sunny,2,15
1,1,qp02z9,49,Residential,1,Not Allowed,No,6.476213,Snowy,2,15
2,2,qp02yf,49,Residential,3,Allowed,Yes,22.318203,Sunny,2,15
3,3,qp02z6,49,Residential,2,Not Allowed,Yes,16.457339,Rainy,2,15
4,4,qp02zd,49,Residential,1,Not Allowed,No,18.266162,Foggy,2,15


### Feature Engineering: Encoding Categorical Features

Applying one-hot encoding to categorical columns: `RoadType`, `LargeVehicles`, `Landmarks`, and `Weather`.

In [ ]:
if not train_df.empty and not test_df.empty:
    categorical_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

    # Apply one-hot encoding
    train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
    test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

    print("Categorical columns one-hot encoded.")

    print("\n--- Train Data Head after encoding ---")
    display(train_df.head())
    print("\n--- Test Data Head after encoding ---")
    display(test_df.head())
    print("\n--- Train Data Info after encoding ---")
    display(train_df.info())
    print("\n--- Test Data Info after encoding ---")
    display(test_df.info())

Categorical columns one-hot encoded.

--- Train Data Head after encoding ---


,Index,geohash,day,demand,NumberofLanes,Temperature,hour,minute,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny
0,0,qp02z1,48,0.048804,1,16.405354,0,0,True,False,True,False,False,False,True
1,1,qp02zt,48,0.118507,3,31.104565,0,0,True,False,False,True,False,False,True
2,2,qp08bj,48,0.027132,1,25.919267,0,0,True,False,True,False,False,False,True
3,3,qp08gt,48,0.003272,1,16.405354,0,0,True,False,True,False,True,False,False
4,4,qp02zq,48,0.010819,1,10.803667,0,0,True,False,True,False,True,False,False



--- Test Data Head after encoding ---


,Index,geohash,day,NumberofLanes,Temperature,hour,minute,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny
0,0,qp02z1,49,1,16.457339,2,15,True,False,True,False,False,False,True
1,1,qp02z9,49,1,6.476213,2,15,True,False,True,False,False,True,False
2,2,qp02yf,49,3,22.318203,2,15,True,False,False,True,False,False,True
3,3,qp02z6,49,2,16.457339,2,15,True,False,True,True,True,False,False
4,4,qp02zd,49,1,18.266162,2,15,True,False,True,False,False,False,False



--- Train Data Info after encoding ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Index                      77299 non-null  int64  
 1   geohash                    77299 non-null  object 
 2   day                        77299 non-null  int64  
 3   demand                     77299 non-null  float64
 4   NumberofLanes              77299 non-null  int64  
 5   Temperature                77299 non-null  float64
 6   hour                       77299 non-null  int32  
 7   minute                     77299 non-null  int32  
 8   RoadType_Residential       77299 non-null  bool   
 9   RoadType_Street            77299 non-null  bool   
 10  LargeVehicles_Not Allowed  77299 non-null  bool   
 11  Landmarks_Yes              77299 non-null  bool   
 12  Weather_Rainy              77299 non-null  bool   
 13  Weathe

None


--- Test Data Info after encoding ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Index                      41778 non-null  int64  
 1   geohash                    41778 non-null  object 
 2   day                        41778 non-null  int64  
 3   NumberofLanes              41778 non-null  int64  
 4   Temperature                41778 non-null  float64
 5   hour                       41778 non-null  int32  
 6   minute                     41778 non-null  int32  
 7   RoadType_Residential       41778 non-null  bool   
 8   RoadType_Street            41778 non-null  bool   
 9   LargeVehicles_Not Allowed  41778 non-null  bool   
 10  Landmarks_Yes              41778 non-null  bool   
 11  Weather_Rainy              41778 non-null  bool   
 12  Weather_Snowy              41778 non-null  bool   
 13  Weather

None

### Feature Engineering: Encoding Geohash

Applying Label Encoding to the `geohash` column to convert it into a numerical feature. First, we will ensure that both train and test datasets have the same geohash encodings.

In [ ]:
from sklearn.preprocessing import LabelEncoder

if not train_df.empty and not test_df.empty:
    # Combine geohash from both train and test to ensure consistent encoding
    all_geohashes = pd.concat([train_df['geohash'], test_df['geohash']], axis=0).unique()

    le = LabelEncoder()
    le.fit(all_geohashes)

    train_df['geohash_encoded'] = le.transform(train_df['geohash'])
    test_df['geohash_encoded'] = le.transform(test_df['geohash'])

    # Drop original geohash column
    train_df.drop('geohash', axis=1, inplace=True)
    test_df.drop('geohash', axis=1, inplace=True)

    print("Geohash column label encoded and original column dropped.")

    print("\n--- Train Data Head after geohash encoding ---")
    display(train_df.head())
    print("\n--- Test Data Head after geohash encoding ---")
    display(test_df.head())
    print("\n--- Train Data Info after geohash encoding ---")
    display(train_df.info())
    print("\n--- Test Data Info after geohash encoding ---")
    display(test_df.info())

Geohash column label encoded and original column dropped.

--- Train Data Head after geohash encoding ---


,Index,day,demand,NumberofLanes,Temperature,hour,minute,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny,geohash_encoded
0,0,48,0.048804,1,16.405354,0,0,True,False,True,False,False,False,True,4
1,1,48,0.118507,3,31.104565,0,0,True,False,False,True,False,False,True,25
2,2,48,0.027132,1,25.919267,0,0,True,False,True,False,False,False,True,370
3,3,48,0.003272,1,16.405354,0,0,True,False,True,False,True,False,False,418
4,4,48,0.010819,1,10.803667,0,0,True,False,True,False,True,False,False,22



--- Test Data Head after geohash encoding ---


,Index,day,NumberofLanes,Temperature,hour,minute,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny,geohash_encoded
0,0,49,1,16.457339,2,15,True,False,True,False,False,False,True,4
1,1,49,1,6.476213,2,15,True,False,True,False,False,True,False,10
2,2,49,3,22.318203,2,15,True,False,False,True,False,False,True,1
3,3,49,2,16.457339,2,15,True,False,True,True,True,False,False,8
4,4,49,1,18.266162,2,15,True,False,True,False,False,False,False,12



--- Train Data Info after geohash encoding ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Index                      77299 non-null  int64  
 1   day                        77299 non-null  int64  
 2   demand                     77299 non-null  float64
 3   NumberofLanes              77299 non-null  int64  
 4   Temperature                77299 non-null  float64
 5   hour                       77299 non-null  int32  
 6   minute                     77299 non-null  int32  
 7   RoadType_Residential       77299 non-null  bool   
 8   RoadType_Street            77299 non-null  bool   
 9   LargeVehicles_Not Allowed  77299 non-null  bool   
 10  Landmarks_Yes              77299 non-null  bool   
 11  Weather_Rainy              77299 non-null  bool   
 12  Weather_Snowy              77299 non-null  bool   
 13

None


--- Test Data Info after geohash encoding ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Index                      41778 non-null  int64  
 1   day                        41778 non-null  int64  
 2   NumberofLanes              41778 non-null  int64  
 3   Temperature                41778 non-null  float64
 4   hour                       41778 non-null  int32  
 5   minute                     41778 non-null  int32  
 6   RoadType_Residential       41778 non-null  bool   
 7   RoadType_Street            41778 non-null  bool   
 8   LargeVehicles_Not Allowed  41778 non-null  bool   
 9   Landmarks_Yes              41778 non-null  bool   
 10  Weather_Rainy              41778 non-null  bool   
 11  Weather_Snowy              41778 non-null  bool   
 12  Weather_Sunny              41778 non-null  bool   
 13 

None

### Log Transform Target Variable

Applying a log transformation (`np.log1p`) to the `demand` column. This can help normalize the distribution of the target variable, which often leads to better model performance.

In [ ]:
if not train_df.empty:
    # Apply log1p transformation to handle potential zeros in demand
    train_df['demand'] = np.log1p(train_df['demand'])
    print("Demand column log-transformed using np.log1p.")

    print("\n--- Train Data Head after log transformation ---")
    display(train_df.head())
else:
    print("Training data (train_df) not found. Please ensure previous steps were executed.")

Demand column log-transformed using np.log1p.

--- Train Data Head after log transformation ---


,Index,day,demand,NumberofLanes,Temperature,hour,minute,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny,geohash_encoded
0,0,48,0.047651,1,16.405354,0,0,True,False,True,False,False,False,True,4
1,1,48,0.111995,3,31.104565,0,0,True,False,False,True,False,False,True,25
2,2,48,0.026770,1,25.919267,0,0,True,False,True,False,False,False,True,370
3,3,48,0.003266,1,16.405354,0,0,True,False,True,False,True,False,False,418
4,4,48,0.010761,1,10.803667,0,0,True,False,True,False,True,False,False,22


### Feature Engineering: Cyclical Features

Creating cyclical features for 'day', 'hour', and 'minute' using sine and cosine transformations. This helps the model understand the cyclic nature of these features (e.g., that 23:00 is close to 00:00).

In [ ]:
if not train_df.empty and not test_df.empty:
    # Day of week (assuming 'day' is 0-6 or similar, or using max_day as period)
    # Assuming 'day' column represents day number and we need to treat it cyclically.
    # Let's assume a 7-day cycle for 'day'. If 'day' is day of year, we'd use 365.
    max_day = train_df['day'].max() + 1 # Assuming 'day' starts from 0 or 1.

    train_df['day_sin'] = np.sin(2 * np.pi * train_df['day'] / max_day)
    train_df['day_cos'] = np.cos(2 * np.pi * train_df['day'] / max_day)
    test_df['day_sin'] = np.sin(2 * np.pi * test_df['day'] / max_day)
    test_df['day_cos'] = np.cos(2 * np.pi * test_df['day'] / max_day)

    # Hour (24-hour cycle)
    train_df['hour_sin'] = np.sin(2 * np.pi * train_df['hour'] / 24)
    train_df['hour_cos'] = np.cos(2 * np.pi * train_df['hour'] / 24)
    test_df['hour_sin'] = np.sin(2 * np.pi * test_df['hour'] / 24)
    test_df['hour_cos'] = np.cos(2 * np.pi * test_df['hour'] / 24)

    # Minute (60-minute cycle)
    train_df['minute_sin'] = np.sin(2 * np.pi * train_df['minute'] / 60)
    train_df['minute_cos'] = np.cos(2 * np.pi * train_df['minute'] / 60)
    test_df['minute_sin'] = np.sin(2 * np.pi * test_df['minute'] / 60)
    test_df['minute_cos'] = np.cos(2 * np.pi * test_df['minute'] / 60)

    # Optionally drop original 'day', 'hour', 'minute' columns if they are fully captured by sin/cos
    # For now, let's keep them and let the model decide.

    print("Cyclical features created for 'day', 'hour', and 'minute'.")

    print("\n--- Train Data Head after cyclical features ---")
    display(train_df.head())
    print("\n--- Test Data Head after cyclical features ---")
    display(test_df.head())

    print("\n--- Train Data Info after cyclical features ---")
    display(train_df.info())
    print("\n--- Test Data Info after cyclical features ---")
    display(test_df.info())
else:
    print("Training or testing data not found. Please ensure previous steps were executed.")

Cyclical features created for 'day', 'hour', and 'minute'.

--- Train Data Head after cyclical features ---


,Index,day,demand,NumberofLanes,Temperature,hour,minute,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,...,Weather_Rainy,Weather_Snowy,Weather_Sunny,geohash_encoded,day_sin,day_cos,hour_sin,hour_cos,minute_sin,minute_cos
0,0,48,0.047651,1,16.405354,0,0,True,False,True,...,False,False,True,4,-0.24869,0.968583,0.0,1.0,0.0,1.0
1,1,48,0.111995,3,31.104565,0,0,True,False,False,...,False,False,True,25,-0.24869,0.968583,0.0,1.0,0.0,1.0
2,2,48,0.026770,1,25.919267,0,0,True,False,True,...,False,False,True,370,-0.24869,0.968583,0.0,1.0,0.0,1.0
3,3,48,0.003266,1,16.405354,0,0,True,False,True,...,True,False,False,418,-0.24869,0.968583,0.0,1.0,0.0,1.0
4,4,48,0.010761,1,10.803667,0,0,True,False,True,...,True,False,False,22,-0.24869,0.968583,0.0,1.0,0.0,1.0



--- Test Data Head after cyclical features ---


,Index,day,NumberofLanes,Temperature,hour,minute,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny,geohash_encoded,day_sin,day_cos,hour_sin,hour_cos,minute_sin,minute_cos
0,0,49,1,16.457339,2,15,True,False,True,False,False,False,True,4,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16
1,1,49,1,6.476213,2,15,True,False,True,False,False,True,False,10,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16
2,2,49,3,22.318203,2,15,True,False,False,True,False,False,True,1,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16
3,3,49,2,16.457339,2,15,True,False,True,True,True,False,False,8,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16
4,4,49,1,18.266162,2,15,True,False,True,False,False,False,False,12,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16



--- Train Data Info after cyclical features ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Index                      77299 non-null  int64  
 1   day                        77299 non-null  int64  
 2   demand                     77299 non-null  float64
 3   NumberofLanes              77299 non-null  int64  
 4   Temperature                77299 non-null  float64
 5   hour                       77299 non-null  int32  
 6   minute                     77299 non-null  int32  
 7   RoadType_Residential       77299 non-null  bool   
 8   RoadType_Street            77299 non-null  bool   
 9   LargeVehicles_Not Allowed  77299 non-null  bool   
 10  Landmarks_Yes              77299 non-null  bool   
 11  Weather_Rainy              77299 non-null  bool   
 12  Weather_Snowy              77299 non-null  bool   
 1

None


--- Test Data Info after cyclical features ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Index                      41778 non-null  int64  
 1   day                        41778 non-null  int64  
 2   NumberofLanes              41778 non-null  int64  
 3   Temperature                41778 non-null  float64
 4   hour                       41778 non-null  int32  
 5   minute                     41778 non-null  int32  
 6   RoadType_Residential       41778 non-null  bool   
 7   RoadType_Street            41778 non-null  bool   
 8   LargeVehicles_Not Allowed  41778 non-null  bool   
 9   Landmarks_Yes              41778 non-null  bool   
 10  Weather_Rainy              41778 non-null  bool   
 11  Weather_Snowy              41778 non-null  bool   
 12  Weather_Sunny              41778 non-null  bool   
 13

None

### Feature Engineering: Interaction Terms

Creating interaction terms between potentially related features to capture more complex relationships that might influence demand. For example, the effect of temperature might be different at different hours.

In [ ]:
if not train_df.empty and not test_df.empty:
    # Temperature and Time interactions
    train_df['Temp_x_Hour'] = train_df['Temperature'] * train_df['hour']
    test_df['Temp_x_Hour'] = test_df['Temperature'] * test_df['hour']

    train_df['Temp_x_Minute'] = train_df['Temperature'] * train_df['minute']
    test_df['Temp_x_Minute'] = test_df['Temperature'] * test_df['minute']

    # Lanes and Geohash interaction
    train_df['Lanes_x_Geohash'] = train_df['NumberofLanes'] * train_df['geohash_encoded']
    test_df['Lanes_x_Geohash'] = test_df['NumberofLanes'] * test_df['geohash_encoded']

    print("Interaction features created: 'Temp_x_Hour', 'Temp_x_Minute', 'Lanes_x_Geohash'.")

    print("\n--- Train Data Head after interaction features ---")
    display(train_df.head())
    print("\n--- Test Data Head after interaction features ---")
    display(test_df.head())

    print("\n--- Train Data Info after interaction features ---")
    display(train_df.info())
    print("\n--- Test Data Info after interaction features ---")
    display(test_df.info())
else:
    print("Training or testing data not found. Please ensure previous steps were executed.")

Interaction features created: 'Temp_x_Hour', 'Temp_x_Minute', 'Lanes_x_Geohash'.

--- Train Data Head after interaction features ---


,Index,day,demand,NumberofLanes,Temperature,hour,minute,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,...,geohash_encoded,day_sin,day_cos,hour_sin,hour_cos,minute_sin,minute_cos,Temp_x_Hour,Temp_x_Minute,Lanes_x_Geohash
0,0,48,0.047651,1,16.405354,0,0,True,False,True,...,4,-0.24869,0.968583,0.0,1.0,0.0,1.0,0.0,0.0,4
1,1,48,0.111995,3,31.104565,0,0,True,False,False,...,25,-0.24869,0.968583,0.0,1.0,0.0,1.0,0.0,0.0,75
2,2,48,0.026770,1,25.919267,0,0,True,False,True,...,370,-0.24869,0.968583,0.0,1.0,0.0,1.0,0.0,0.0,370
3,3,48,0.003266,1,16.405354,0,0,True,False,True,...,418,-0.24869,0.968583,0.0,1.0,0.0,1.0,0.0,0.0,418
4,4,48,0.010761,1,10.803667,0,0,True,False,True,...,22,-0.24869,0.968583,0.0,1.0,0.0,1.0,0.0,0.0,22



--- Test Data Head after interaction features ---


,Index,day,NumberofLanes,Temperature,hour,minute,RoadType_Residential,RoadType_Street,LargeVehicles_Not Allowed,Landmarks_Yes,...,geohash_encoded,day_sin,day_cos,hour_sin,hour_cos,minute_sin,minute_cos,Temp_x_Hour,Temp_x_Minute,Lanes_x_Geohash
0,0,49,1,16.457339,2,15,True,False,True,False,...,4,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16,32.914677,246.860078,4
1,1,49,1,6.476213,2,15,True,False,True,False,...,10,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16,12.952427,97.143202,10
2,2,49,3,22.318203,2,15,True,False,False,True,...,1,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16,44.636405,334.773039,3
3,3,49,2,16.457339,2,15,True,False,True,True,...,8,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16,32.914677,246.860078,16
4,4,49,1,18.266162,2,15,True,False,True,False,...,12,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16,36.532324,273.992433,12



--- Train Data Info after interaction features ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Index                      77299 non-null  int64  
 1   day                        77299 non-null  int64  
 2   demand                     77299 non-null  float64
 3   NumberofLanes              77299 non-null  int64  
 4   Temperature                77299 non-null  float64
 5   hour                       77299 non-null  int32  
 6   minute                     77299 non-null  int32  
 7   RoadType_Residential       77299 non-null  bool   
 8   RoadType_Street            77299 non-null  bool   
 9   LargeVehicles_Not Allowed  77299 non-null  bool   
 10  Landmarks_Yes              77299 non-null  bool   
 11  Weather_Rainy              77299 non-null  bool   
 12  Weather_Snowy              77299 non-null  bool   

None


--- Test Data Info after interaction features ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Index                      41778 non-null  int64  
 1   day                        41778 non-null  int64  
 2   NumberofLanes              41778 non-null  int64  
 3   Temperature                41778 non-null  float64
 4   hour                       41778 non-null  int32  
 5   minute                     41778 non-null  int32  
 6   RoadType_Residential       41778 non-null  bool   
 7   RoadType_Street            41778 non-null  bool   
 8   LargeVehicles_Not Allowed  41778 non-null  bool   
 9   Landmarks_Yes              41778 non-null  bool   
 10  Weather_Rainy              41778 non-null  bool   
 11  Weather_Snowy              41778 non-null  bool   
 12  Weather_Sunny              41778 non-null  bool   


None

### Prepare Data for Model Training

Separating features (X) and target (y) from the training data, and splitting it into training and validation sets. Ensuring column consistency between train and test sets.

In [ ]:
if not train_df.empty and not test_df.empty:
    # Define features (X) and target (y)
    X = train_df.drop(['Index', 'demand'], axis=1)
    y = train_df['demand']

    # Keep 'Index' for test_df for submission
    test_ids = test_df['Index']
    X_test = test_df.drop('Index', axis=1)

    # Align columns - very important after one-hot encoding
    train_cols = X.columns
    test_cols = X_test.columns

    missing_in_test = set(train_cols) - set(test_cols)
    for c in missing_in_test:
        X_test[c] = 0

    missing_in_train = set(test_cols) - set(train_cols)
    for c in missing_in_train:
        X[c] = 0

    X_test = X_test[train_cols] # Ensure the order of columns is the same

    # Convert boolean columns to int to avoid LightGBM feature mismatch issues
    for col in X.select_dtypes(include='bool').columns:
        X[col] = X[col].astype(int)
    for col in X_test.select_dtypes(include='bool').columns:
        X_test[col] = X_test[col].astype(int)

    print("Features and target separated and boolean columns converted to integers.")
    print(f"X shape: {X.shape}, y shape: {y.shape}")
    print(f"X_test shape: {X_test.shape}")

    # Split the training data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # Convert boolean columns in X_train and X_val to int as well
    for col in X_train.select_dtypes(include='bool').columns:
        X_train[col] = X_train[col].astype(int)
    for col in X_val.select_dtypes(include='bool').columns:
        X_val[col] = X_val[col].astype(int)

    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")

Features and target separated and boolean columns converted to integers.
X shape: (77299, 22), y shape: (77299,)
X_test shape: (41778, 22)
X_train shape: (61839, 22), y_train shape: (61839,)
X_val shape: (15460, 22), y_val shape: (15460,)


### Model Training: LightGBM Regressor

Initializing and training a LightGBM Regressor model on the `X_train` and `y_train` datasets.

In [ ]:
if 'X_train' in locals() and 'y_train' in locals():
    lgb_reg = lgb.LGBMRegressor(random_state=42)
    lgb_reg.fit(X_train, y_train)
    print("LightGBM Regressor model trained successfully.")
else:
    print("Training data (X_train, y_train) not found. Please ensure previous steps were executed.")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003078 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 561
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 13
[LightGBM] [Info] Start training from score 0.093784
LightGBM Regressor model trained successfully.


### Model Evaluation

Predicting on the validation set and calculating R2 score and Mean Squared Error to evaluate the model's performance.

In [ ]:
if 'lgb_reg' in locals() and 'X_val' in locals() and 'y_val' in locals():
    y_pred_val = lgb_reg.predict(X_val)

    r2 = r2_score(y_val, y_pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))

    print(f"Validation R2 Score: {r2:.4f}")
    print(f"Validation RMSE: {rmse:.4f}")

    if r2 >= 0.95:
        print("Achieved 95% R2 score on the validation set!")
    else:
        print("R2 score is below 95%. Further hyperparameter tuning or feature engineering might be needed.")
else:
    print("Model or validation data not found. Please ensure previous steps were executed.")

Validation R2 Score: 0.8493
Validation RMSE: 0.0552
R2 score is below 95%. Further hyperparameter tuning or feature engineering might be needed.


### Make Predictions on Test Data and Generate Submission File

Using the trained LightGBM model to predict demand on the `X_test` dataset and creating a `submission.csv` file.

In [ ]:
if 'lgb_reg' in locals() and 'X_test' in locals() and 'test_ids' in locals():
    y_pred_test = lgb_reg.predict(X_test)

    # Ensure predictions are non-negative
    y_pred_test[y_pred_test < 0] = 0

    submission_df = pd.DataFrame({'Index': test_ids, 'demand': y_pred_test})
    submission_df.to_csv('submission.csv', index=False)

    print("Predictions made on the test set.")
    print("Submission file 'submission.csv' created successfully!")
    display(submission_df.head())
else:
    print("Model, test data, or test IDs not found. Please ensure previous steps were executed correctly.")

Predictions made on the test set.
Submission file 'submission.csv' created successfully!


,Index,demand
0,0,0.043280
1,1,0.043280
2,2,0.043246
3,3,0.043318
4,4,0.042919


### Hyperparameter Tuning with GridSearchCV

To improve the model's performance and achieve a higher R2 score, I'll use `GridSearchCV` to find optimal hyperparameters for the LightGBM Regressor.

In [ ]:
from sklearn.model_selection import GridSearchCV

if 'X_train' in locals() and 'y_train' in locals():
    # Define the parameter grid to search
    param_grid = {
        'n_estimators': [100, 200, 300],  # Number of boosting rounds
        'learning_rate': [0.01, 0.05, 0.1],  # Step size shrinkage
        'num_leaves': [20, 31, 40],  # Max number of leaves in one tree
        'max_depth': [5, 7, -1],  # Max tree depth, -1 means no limit
        'reg_alpha': [0.1, 0.5],  # L1 regularization
        'reg_lambda': [0.1, 0.5]  # L2 regularization
    }

    lgb_tuned = lgb.LGBMRegressor(random_state=42)

    # Initialize GridSearchCV
    grid_search = GridSearchCV(estimator=lgb_tuned, param_grid=param_grid,
                               cv=3, scoring='r2', verbose=1, n_jobs=-1)

    # Fit GridSearchCV to the training data
    print("Starting GridSearchCV for hyperparameter tuning...")
    grid_search.fit(X_train, y_train)

    print("GridSearchCV completed.")
    print(f"Best parameters found: {grid_search.best_params_}")
    print(f"Best R2 score (cross-validated): {grid_search.best_score_:.4f}")

    # Update the model with the best parameters
    lgb_reg_optimized = grid_search.best_estimator_
    print("Optimized LightGBM Regressor model created.")
else:
    print("Training data (X_train, y_train) not found. Please ensure previous steps were executed.")

### Evaluate Optimized Model

Predicting on the validation set with the optimized model and recalculating R2 score and Mean Squared Error.

In [ ]:
if 'lgb_reg_optimized' in locals() and 'X_val' in locals() and 'y_val' in locals():
    y_pred_val_optimized = lgb_reg_optimized.predict(X_val)

    r2_optimized = r2_score(y_val, y_pred_val_optimized)
    rmse_optimized = np.sqrt(mean_squared_error(y_val, y_pred_val_optimized))

    print(f"Optimized Validation R2 Score: {r2_optimized:.4f}")
    print(f"Optimized Validation RMSE: {rmse_optimized:.4f}")

    if r2_optimized >= 0.95:
        print("Achieved 95% R2 score on the validation set with optimized model!")
    else:
        print("R2 score is still below 95% after optimization. Further tuning or feature engineering might be needed.")

    # Update the lgb_reg for final predictions to the optimized one
    lgb_reg = lgb_reg_optimized
    print("Current prediction model updated to optimized version.")
else:
    print("Optimized model or validation data not found. Please ensure previous steps were executed.")

### Make Predictions on Test Data with Optimized Model and Generate Submission File

Using the optimized LightGBM model to predict demand on the `X_test` dataset and creating a new `submission_optimized.csv` file.

In [ ]:
if 'lgb_reg' in locals() and 'X_test' in locals() and 'test_ids' in locals():
    y_pred_test_optimized = lgb_reg.predict(X_test)

    # Ensure predictions are non-negative
    y_pred_test_optimized[y_pred_test_optimized < 0] = 0

    submission_df_optimized = pd.DataFrame({'Index': test_ids, 'demand': y_pred_test_optimized})
    submission_df_optimized.to_csv('submission_optimized.csv', index=False)

    print("Predictions made on the test set with optimized model.")
    print("Submission file 'submission_optimized.csv' created successfully!")
    display(submission_df_optimized.head())
else:
    print("Optimized model, test data, or test IDs not found. Please ensure previous steps were executed correctly.")

### Hyperparameter Tuning with GridSearchCV

To improve the model's performance and achieve a higher R2 score, I'll use `GridSearchCV` to find optimal hyperparameters for the LightGBM Regressor.

In [ ]:
from sklearn.model_selection import GridSearchCV

if 'X_train' in locals() and 'y_train' in locals():
    # Define the parameter grid to search
    param_grid = {
        'n_estimators': [100, 200, 300],  # Number of boosting rounds
        'learning_rate': [0.01, 0.05, 0.1],  # Step size shrinkage
        'num_leaves': [20, 31, 40],  # Max number of leaves in one tree
        'max_depth': [5, 7, -1],  # Max tree depth, -1 means no limit
        'reg_alpha': [0.1, 0.5],  # L1 regularization
        'reg_lambda': [0.1, 0.5]  # L2 regularization
    }

    lgb_tuned = lgb.LGBMRegressor(random_state=42)

    # Initialize GridSearchCV
    grid_search = GridSearchCV(estimator=lgb_tuned, param_grid=param_grid,
                               cv=3, scoring='r2', verbose=1, n_jobs=-1)

    # Fit GridSearchCV to the training data
    print("Starting GridSearchCV for hyperparameter tuning...")
    grid_search.fit(X_train, y_train)

    print("GridSearchCV completed.")
    print(f"Best parameters found: {grid_search.best_params_}")
    print(f"Best R2 score (cross-validated): {grid_search.best_score_:.4f}")

    # Update the model with the best parameters
    lgb_reg_optimized = grid_search.best_estimator_
    print("Optimized LightGBM Regressor model created.")
else:
    print("Training data (X_train, y_train) not found. Please ensure previous steps were executed.")

### Evaluate Optimized Model

Predicting on the validation set with the optimized model and recalculating R2 score and Mean Squared Error.

In [ ]:
if 'lgb_reg_optimized' in locals() and 'X_val' in locals() and 'y_val' in locals():
    y_pred_val_optimized = lgb_reg_optimized.predict(X_val)

    r2_optimized = r2_score(y_val, y_pred_val_optimized)
    rmse_optimized = np.sqrt(mean_squared_error(y_val, y_pred_val_optimized))

    print(f"Optimized Validation R2 Score: {r2_optimized:.4f}")
    print(f"Optimized Validation RMSE: {rmse_optimized:.4f}")

    if r2_optimized >= 0.95:
        print("Achieved 95% R2 score on the validation set with optimized model!")
    else:
        print("R2 score is still below 95% after optimization. Further tuning or feature engineering might be needed.")

    # Update the lgb_reg for final predictions to the optimized one
    lgb_reg = lgb_reg_optimized
    print("Current prediction model updated to optimized version.")
else:
    print("Optimized model or validation data not found. Please ensure previous steps were executed.")

### Make Predictions on Test Data with Optimized Model and Generate Submission File

Using the optimized LightGBM model to predict demand on the `X_test` dataset and creating a new `submission_optimized.csv` file.

In [ ]:
if 'lgb_reg' in locals() and 'X_test' in locals() and 'test_ids' in locals():
    y_pred_test_optimized = lgb_reg.predict(X_test)

    # Ensure predictions are non-negative
    y_pred_test_optimized[y_pred_test_optimized < 0] = 0

    submission_df_optimized = pd.DataFrame({'Index': test_ids, 'demand': y_pred_test_optimized})
    submission_df_optimized.to_csv('submission_optimized.csv', index=False)

    print("Predictions made on the test set with optimized model.")
    print("Submission file 'submission_optimized.csv' created successfully!")
    display(submission_df_optimized.head())
else:
    print("Optimized model, test data, or test IDs not found. Please ensure previous steps were executed correctly.")

### Hyperparameter Tuning with GridSearchCV

To improve the model's performance and achieve a higher R2 score, I'll use `GridSearchCV` to find optimal hyperparameters for the LightGBM Regressor.

In [ ]:
from sklearn.model_selection import GridSearchCV

if 'X_train' in locals() and 'y_train' in locals():
    # Define the parameter grid to search
    param_grid = {
        'n_estimators': [100, 200, 300],  # Number of boosting rounds
        'learning_rate': [0.01, 0.05, 0.1],  # Step size shrinkage
        'num_leaves': [20, 31, 40],  # Max number of leaves in one tree
        'max_depth': [5, 7, -1],  # Max tree depth, -1 means no limit
        'reg_alpha': [0.1, 0.5],  # L1 regularization
        'reg_lambda': [0.1, 0.5]  # L2 regularization
    }

    lgb_tuned = lgb.LGBMRegressor(random_state=42)

    # Initialize GridSearchCV
    grid_search = GridSearchCV(estimator=lgb_tuned, param_grid=param_grid,
                               cv=3, scoring='r2', verbose=1, n_jobs=-1)

    # Fit GridSearchCV to the training data
    print("Starting GridSearchCV for hyperparameter tuning...")
    grid_search.fit(X_train, y_train)

    print("GridSearchCV completed.")
    print(f"Best parameters found: {grid_search.best_params_}")
    print(f"Best R2 score (cross-validated): {grid_search.best_score_:.4f}")

    # Update the model with the best parameters
    lgb_reg_optimized = grid_search.best_estimator_
    print("Optimized LightGBM Regressor model created.")
else:
    print("Training data (X_train, y_train) not found. Please ensure previous steps were executed.")

Starting GridSearchCV for hyperparameter tuning...
Fitting 3 folds for each of 324 candidates, totalling 972 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004299 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1381
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 22
[LightGBM] [Info] Start training from score 0.083012
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
GridSearchCV completed.
Best parameters found: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300, 'num_leaves': 40, 'reg_alpha': 0.1, 'reg_lambda': 0.1}
Best R2 score

### Evaluate Optimized Model

Predicting on the validation set with the optimized model and recalculating R2 score and Mean Squared Error.

In [ ]:
if 'lgb_reg_optimized' in locals() and 'X_val' in locals() and 'y_val' in locals():
    y_pred_val_optimized = lgb_reg_optimized.predict(X_val)

    r2_optimized = r2_score(y_val, y_pred_val_optimized)
    rmse_optimized = np.sqrt(mean_squared_error(y_val, y_pred_val_optimized))

    print(f"Optimized Validation R2 Score: {r2_optimized:.4f}")
    print(f"Optimized Validation RMSE: {rmse_optimized:.4f}")

    if r2_optimized >= 0.95:
        print("Achieved 95% R2 score on the validation set with optimized model!")
    else:
        print("R2 score is still below 95% after optimization. Further tuning or feature engineering might be needed.")

    # Update the lgb_reg for final predictions to the optimized one
    lgb_reg = lgb_reg_optimized
    print("Current prediction model updated to optimized version.")
else:
    print("Optimized model or validation data not found. Please ensure previous steps were executed.")

Optimized Validation R2 Score: 0.8430
Optimized Validation RMSE: 0.0433
R2 score is still below 95% after optimization. Further tuning or feature engineering might be needed.
Current prediction model updated to optimized version.


### Make Predictions on Test Data with Optimized Model and Generate Submission File

Using the optimized LightGBM model to predict demand on the `X_test` dataset and creating a new `submission_optimized.csv` file.

In [ ]:
if 'lgb_reg' in locals() and 'X_test' in locals() and 'test_ids' in locals():
    y_pred_test_optimized = lgb_reg.predict(X_test)

    # The demand was log-transformed, so inverse transform the predictions
    # np.expm1 is the inverse of np.log1p
    y_pred_test_optimized = np.expm1(y_pred_test_optimized)

    # Ensure predictions are non-negative after inverse transformation
    y_pred_test_optimized[y_pred_test_optimized < 0] = 0

    submission_df_optimized = pd.DataFrame({'Index': test_ids, 'demand': y_pred_test_optimized})
    submission_df_optimized.to_csv('submission_optimized.csv', index=False)

    print("Predictions made on the test set with optimized model.")
    print("Submission file 'submission_optimized.csv' created successfully!")
    display(submission_df_optimized.head())
else:
    print("Optimized model, test data, or test IDs not found. Please ensure previous steps were executed correctly.")

Predictions made on the test set with optimized model.
Submission file 'submission_optimized.csv' created successfully!


,Index,demand
0,0,0.040468
1,1,0.027085
2,2,0.032258
3,3,0.040468
4,4,0.028095


### Feature Engineering: Polynomial Features

Creating polynomial features can help the model capture non-linear relationships and interactions between existing features that might be influencing demand in more complex ways. I'll generate polynomial features for selected numerical columns (excluding cyclical and boolean features) to expand the feature set.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

if 'X' in locals() and 'X_test' in locals() and 'y' in locals():
    # Identify numerical columns for polynomial features
    numerical_cols_for_poly = ['day', 'NumberofLanes', 'Temperature', 'hour', 'minute',
                               'geohash_encoded', 'Temp_x_Hour', 'Temp_x_Minute', 'Lanes_x_Geohash']

    # Identify other columns to keep as is (cyclical and boolean features)
    other_cols = [col for col in X.columns if col not in numerical_cols_for_poly]

    # Initialize PolynomialFeatures with degree 2 (to capture squared terms and interactions)
    poly = PolynomialFeatures(degree=2, include_bias=False)

    # Apply polynomial features to selected numerical columns in training data
    X_poly_data = poly.fit_transform(X[numerical_cols_for_poly])
    # Create a DataFrame for the new polynomial features
    X_poly_df = pd.DataFrame(X_poly_data, columns=poly.get_feature_names_out(numerical_cols_for_poly), index=X.index)

    # Combine polynomial features with other features for training data
    X_combined_poly = pd.concat([X_poly_df, X[other_cols]], axis=1)

    # Apply polynomial features to selected numerical columns in test data
    X_test_poly_data = poly.transform(X_test[numerical_cols_for_poly]) # Use transform, not fit_transform
    # Create a DataFrame for the new polynomial features in test data
    X_test_poly_df = pd.DataFrame(X_test_poly_data, columns=poly.get_feature_names_out(numerical_cols_for_poly), index=X_test.index)

    # Combine polynomial features with other features for test data
    X_test_combined_poly = pd.concat([X_test_poly_df, X_test[other_cols]], axis=1)

    # Ensure column order is the same for X_combined_poly and X_test_combined_poly
    # This is important for LightGBM and prediction consistency
    X_test_combined_poly = X_test_combined_poly[X_combined_poly.columns]

    print("Polynomial features created and combined with other features.")
    print(f"X_combined_poly shape: {X_combined_poly.shape}")
    print(f"X_test_combined_poly shape: {X_test_combined_poly.shape}")

    print("\n--- X_combined_poly Head ---")
    display(X_combined_poly.head())
    print("\n--- X_combined_poly Info ---")
    display(X_combined_poly.info())

    print("\n--- X_test_combined_poly Head ---")
    display(X_test_combined_poly.head())
    print("\n--- X_test_combined_poly Info ---")
    display(X_test_combined_poly.info())

    # Update X and X_test to the new combined dataframes for subsequent steps
    X = X_combined_poly
    X_test = X_test_combined_poly
else:
    print("X, X_test, or y not found. Please ensure previous data preparation steps were executed.")

Polynomial features created and combined with other features.
X_combined_poly shape: (77299, 112)
X_test_combined_poly shape: (41778, 202)

--- X_combined_poly Head ---


,day,NumberofLanes,Temperature,hour,minute,geohash_encoded,Temp_x_Hour,Temp_x_Minute,Lanes_x_Geohash,day^2,...,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny,day_sin,day_cos,hour_sin,hour_cos,minute_sin,minute_cos
0,48.0,1.0,16.405354,0.0,0.0,4.0,0.0,0.0,4.0,2304.0,...,0,0,0,1,-0.24869,0.968583,0.0,1.0,0.0,1.0
1,48.0,3.0,31.104565,0.0,0.0,25.0,0.0,0.0,75.0,2304.0,...,1,0,0,1,-0.24869,0.968583,0.0,1.0,0.0,1.0
2,48.0,1.0,25.919267,0.0,0.0,370.0,0.0,0.0,370.0,2304.0,...,0,0,0,1,-0.24869,0.968583,0.0,1.0,0.0,1.0
3,48.0,1.0,16.405354,0.0,0.0,418.0,0.0,0.0,418.0,2304.0,...,0,1,0,0,-0.24869,0.968583,0.0,1.0,0.0,1.0
4,48.0,1.0,10.803667,0.0,0.0,22.0,0.0,0.0,22.0,2304.0,...,0,1,0,0,-0.24869,0.968583,0.0,1.0,0.0,1.0



--- X_combined_poly Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Columns: 112 entries, day to minute_cos
dtypes: float64(105), int64(7)
memory usage: 66.1 MB


None


--- X_test_combined_poly Head ---


,day,NumberofLanes,Temperature,hour,minute,geohash_encoded,Temp_x_Hour,Temp_x_Minute,Lanes_x_Geohash,day^2,...,Landmarks_Yes,Weather_Rainy,Weather_Snowy,Weather_Sunny,day_sin,day_cos,hour_sin,hour_cos,minute_sin,minute_cos
0,49.0,1.0,16.457339,2.0,15.0,4.0,32.914677,246.860078,4.0,2401.0,...,0,0,0,1,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16
1,49.0,1.0,6.476213,2.0,15.0,10.0,12.952427,97.143202,10.0,2401.0,...,0,0,1,0,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16
2,49.0,3.0,22.318203,2.0,15.0,1.0,44.636405,334.773039,3.0,2401.0,...,1,0,0,1,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16
3,49.0,2.0,16.457339,2.0,15.0,8.0,32.914677,246.860078,16.0,2401.0,...,1,1,0,0,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16
4,49.0,1.0,18.266162,2.0,15.0,12.0,36.532324,273.992433,12.0,2401.0,...,0,0,0,0,-0.125333,0.992115,0.5,0.866025,1.0,2.832769e-16



--- X_test_combined_poly Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Columns: 202 entries, day to minute_cos
dtypes: float64(195), int64(7)
memory usage: 64.4 MB


None

### Reset X and X_test for Polynomial Feature Engineering

To ensure polynomial feature engineering is applied correctly without accumulating features from previous runs, we need to reset `X` and `X_test` to their state *before* any polynomial transformations. This means re-deriving them from `train_df` and `test_df` after all other (non-polynomial) feature engineering and column alignment.

In [ ]:
if not train_df.empty and not test_df.empty:
    # Define features (X) and target (y) from the latest train_df (which has cyclical and interaction features)
    # This assumes train_df and test_df are in their final state before polynomial features.
    X_base = train_df.drop(['Index', 'demand'], axis=1)
    y = train_df['demand']

    # Keep 'Index' for test_df for submission
    test_ids = test_df['Index']
    X_test_base = test_df.drop('Index', axis=1)

    # Align columns - very important after one-hot encoding if it was run previously
    train_cols_base = X_base.columns
    test_cols_base = X_test_base.columns

    missing_in_test_base = set(train_cols_base) - set(test_cols_base)
    for c in missing_in_test_base:
        X_test_base[c] = 0

    missing_in_train_base = set(test_cols_base) - set(train_cols_base)
    for c in missing_in_train_base:
        X_base[c] = 0

    X_test_base = X_test_base[train_cols_base] # Ensure the order of columns is the same

    # Convert boolean columns to int to avoid LightGBM feature mismatch issues if not already done
    for col in X_base.select_dtypes(include='bool').columns:
        X_base[col] = X_base[col].astype(int)
    for col in X_test_base.select_dtypes(include='bool').columns:
        X_test_base[col] = X_test_base[col].astype(int)

    print("X and X_test reset to base features before polynomial transformation.")
    print(f"X_base shape: {X_base.shape}, y shape: {y.shape}")
    print(f"X_test_base shape: {X_test_base.shape}")

    # Now, assign these base features to X and X_test so that the polynomial feature cell works on them
    X = X_base
    X_test = X_test_base

else:
    print("Training or testing data frames not found. Please ensure previous data loading and processing steps were executed.")

### Reset X and X_test for Polynomial Feature Engineering

To ensure polynomial feature engineering is applied correctly without accumulating features from previous runs, we need to reset `X` and `X_test` to their state *before* any polynomial transformations. This means re-deriving them from `train_df` and `test_df` after all other (non-polynomial) feature engineering and column alignment.

In [ ]:
if not train_df.empty and not test_df.empty:
    # Define features (X) and target (y) from the latest train_df (which has cyclical and interaction features)
    # This assumes train_df and test_df are in their final state before polynomial features.
    X_base = train_df.drop(['Index', 'demand'], axis=1)
    y = train_df['demand']

    # Keep 'Index' for test_df for submission
    test_ids = test_df['Index']
    X_test_base = test_df.drop('Index', axis=1)

    # Align columns - very important after one-hot encoding if it was run previously
    train_cols_base = X_base.columns
    test_cols_base = X_test_base.columns

    missing_in_test_base = set(train_cols_base) - set(test_cols_base)
    for c in missing_in_test_base:
        X_test_base[c] = 0

    missing_in_train_base = set(test_cols_base) - set(train_cols_base)
    for c in missing_in_train_base:
        X_base[c] = 0

    X_test_base = X_test_base[train_cols_base] # Ensure the order of columns is the same

    # Convert boolean columns to int to avoid LightGBM feature mismatch issues if not already done
    for col in X_base.select_dtypes(include='bool').columns:
        X_base[col] = X_base[col].astype(int)
    for col in X_test_base.select_dtypes(include='bool').columns:
        X_test_base[col] = X_test_base[col].astype(int)

    print("X and X_test reset to base features before polynomial transformation.")
    print(f"X_base shape: {X_base.shape}, y shape: {y.shape}")
    print(f"X_test_base shape: {X_test_base.shape}")

    # Now, assign these base features to X and X_test so that the polynomial feature cell works on them
    X = X_base
    X_test = X_test_base

else:
    print("Training or testing data frames not found. Please ensure previous data loading and processing steps were executed.")

### Prepare Data for Model Training (with Polynomial Features)

Now that polynomial features have been generated and combined, I will re-split the training data into training and validation sets using the enhanced feature set. This ensures the model is trained and evaluated on the new features.

In [ ]:
from sklearn.model_selection import train_test_split

if 'X' in locals() and 'y' in locals() and 'X_test' in locals():
    # X and X_test now contain polynomial features

    # Split the training data into training and validation sets
    X_train_poly, X_val_poly, y_train_poly, y_val_poly = train_test_split(X, y, test_size=0.2, random_state=42)

    print("Data split into training and validation sets for polynomial features.")
    print(f"X_train_poly shape: {X_train_poly.shape}, y_train_poly shape: {y_train_poly.shape}")
    print(f"X_val_poly shape: {X_val_poly.shape}, y_val_poly shape: {y_val_poly.shape}")
else:
    print("X, y, or X_test not found. Please ensure polynomial feature engineering was executed.")

Data split into training and validation sets for polynomial features.
X_train_poly shape: (61839, 112), y_train_poly shape: (61839,)
X_val_poly shape: (15460, 112), y_val_poly shape: (15460,)


### Hyperparameter Tuning with GridSearchCV (Polynomial Features)

I will re-run `GridSearchCV` to find the optimal hyperparameters for the LightGBM Regressor using the dataset enhanced with polynomial features. This will allow the model to leverage the new feature interactions effectively.

In [ ]:
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb

if 'X_train_poly' in locals() and 'y_train_poly' in locals():
    # Define the parameter grid to search
    param_grid_poly = {
        'n_estimators': [100, 200, 300],  # Number of boosting rounds
        'learning_rate': [0.01, 0.05, 0.1],  # Step size shrinkage
        'num_leaves': [20, 31, 40],  # Max number of leaves in one tree
        'max_depth': [5, 7, -1],  # Max tree depth, -1 means no limit
        'reg_alpha': [0.1, 0.5],  # L1 regularization
        'reg_lambda': [0.1, 0.5]  # L2 regularization
    }

    lgb_tuned_poly = lgb.LGBMRegressor(random_state=42)

    # Initialize GridSearchCV
    grid_search_poly = GridSearchCV(estimator=lgb_tuned_poly, param_grid=param_grid_poly,
                               cv=3, scoring='r2', verbose=1, n_jobs=-1)

    # Fit GridSearchCV to the training data with polynomial features
    print("Starting GridSearchCV for hyperparameter tuning with polynomial features...")
    grid_search_poly.fit(X_train_poly, y_train_poly)

    print("GridSearchCV completed for polynomial features.")
    print(f"Best parameters found with polynomial features: {grid_search_poly.best_params_}")
    print(f"Best R2 score (cross-validated) with polynomial features: {grid_search_poly.best_score_:.4f}")

    # Update the model with the best parameters
    lgb_reg_poly_optimized = grid_search_poly.best_estimator_
    print("Optimized LightGBM Regressor model with polynomial features created.")
else:
    print("Training data (X_train_poly, y_train_poly) not found. Please ensure polynomial feature data preparation was executed.")

Starting GridSearchCV for hyperparameter tuning with polynomial features...
Fitting 3 folds for each of 324 candidates, totalling 972 fits


ValueError: 
All the 972 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
972 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1398, in fit
    super().fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/engine.py", line 297, in train
    booster = Booster(params=params, train_set=train_set)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/pyth

### Evaluate Optimized Model (Polynomial Features)

I will now evaluate the performance of the LightGBM model that was optimized using polynomial features. This will provide an updated R2 score and RMSE, helping us assess the impact of the new features on model accuracy.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

if 'lgb_reg_poly_optimized' in locals() and 'X_val_poly' in locals() and 'y_val_poly' in locals():
    y_pred_val_poly = lgb_reg_poly_optimized.predict(X_val_poly)

    r2_poly = r2_score(y_val_poly, y_pred_val_poly)
    rmse_poly = np.sqrt(mean_squared_error(y_val_poly, y_pred_val_poly))

    print(f"Optimized Validation R2 Score with Polynomial Features: {r2_poly:.4f}")
    print(f"Optimized Validation RMSE with Polynomial Features: {rmse_poly:.4f}")

    if r2_poly >= 0.95:
        print("Achieved 95% R2 score on the validation set with polynomial features!")
    else:
        print("R2 score is still below 95% after polynomial features. Further improvements might be needed.")
else:
    print("Optimized polynomial model or validation data not found. Please ensure previous steps were executed.")

LightGBMError: The number of features in data (112) is not the same as it was in training data (67).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.

### Make Predictions on Test Data with Optimized Polynomial Model and Generate Submission File

Using the optimized LightGBM model trained with polynomial features, I will now generate predictions on the test dataset and create a new submission file named `submission_polynomial.csv`. This will reflect the final performance with the enhanced feature set.

In [ ]:
import pandas as pd
import numpy as np

if 'lgb_reg_poly_optimized' in locals() and 'X_test' in locals() and 'test_ids' in locals():
    y_pred_test_poly = lgb_reg_poly_optimized.predict(X_test)

    # The demand was log-transformed, so inverse transform the predictions
    y_pred_test_poly = np.expm1(y_pred_test_poly)

    # Ensure predictions are non-negative after inverse transformation
    y_pred_test_poly[y_pred_test_poly < 0] = 0

    submission_df_poly = pd.DataFrame({'Index': test_ids, 'demand': y_pred_test_poly})
    submission_df_poly.to_csv('submission_polynomial.csv', index=False) # New filename

    print("Predictions made on the test set with optimized polynomial features model.")
    print("Submission file 'submission_polynomial.csv' created successfully!")
    display(submission_df_poly.head())
else:
    print("Optimized polynomial model, test data, or test IDs not found. Please ensure previous steps were executed correctly.")

LightGBMError: The number of features in data (202) is not the same as it was in training data (67).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.

### Feature Engineering: Polynomial Features

Creating polynomial features can help the model capture non-linear relationships and interactions between existing features that might be influencing demand in more complex ways. I'll generate polynomial features for selected numerical columns (excluding cyclical and boolean features) to expand the feature set.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
import re
import pandas as pd # Ensure pandas is imported for DataFrame operations

def clean_feature_names(names):
    cleaned_names = []
    seen_names = set()
    name_counts = {}

    for name in names:
        # Replace '^' with '_pow_'
        name = re.sub(r'(\w+)\^(\d+)', r'\1_pow_\2', name)
        # Replace spaces and other problematic characters with underscores
        name = re.sub(r'[^a-zA-Z0-9_]', '_', name)
        # Remove trailing/leading underscores
        name = name.strip('_')
        # Replace multiple underscores with a single underscore
        name = re.sub(r'_+', '_', name)

        # Ensure uniqueness by appending a number if name already exists
        original_name = name
        count = name_counts.get(original_name, 0)
        while name in seen_names:
            count += 1
            name = f"{original_name}_{count}"
        seen_names.add(name)
        name_counts[original_name] = count
        cleaned_names.append(name)
    return cleaned_names

if 'X' in locals() and 'X_test' in locals() and 'y' in locals():
    # Identify numerical columns for polynomial features
    numerical_cols_for_poly = ['day', 'NumberofLanes', 'Temperature', 'hour', 'minute',
                               'geohash_encoded', 'Temp_x_Hour', 'Temp_x_Minute', 'Lanes_x_Geohash']

    # Identify other columns to keep as is (cyclical and boolean features)
    other_cols = [col for col in X.columns if col not in numerical_cols_for_poly]

    # Initialize PolynomialFeatures with degree 2 (to capture squared terms and interactions)
    poly = PolynomialFeatures(degree=2, include_bias=False)

    # Fit PolynomialFeatures only on the training numerical columns
    poly.fit(X[numerical_cols_for_poly])

    # Generate and clean feature names ONCE, based on the fitted polynomial features
    raw_poly_feature_names = poly.get_feature_names_out(numerical_cols_for_poly)
    poly_feature_names = clean_feature_names(raw_poly_feature_names)

    # Apply polynomial features to selected numerical columns in training data
    X_poly_data = poly.transform(X[numerical_cols_for_poly])
    # Create a DataFrame for the new polynomial features, using the cleaned names
    X_poly_df = pd.DataFrame(X_poly_data, columns=poly_feature_names, index=X.index)

    # Apply polynomial features to selected numerical columns in test data
    X_test_poly_data = poly.transform(X_test[numerical_cols_for_poly]) # Use transform, not fit_transform
    # Create a DataFrame for the new polynomial features in test data, using the SAME cleaned names
    X_test_poly_df = pd.DataFrame(X_test_poly_data, columns=poly_feature_names, index=X_test.index)

    # Combine polynomial features with other features for training data
    X_combined_poly = pd.concat([X_poly_df, X[other_cols]], axis=1)

    # Combine polynomial features with other features for test data
    X_test_combined_poly = pd.concat([X_test_poly_df, X_test[other_cols]], axis=1)

    # Ensure column order is the same for X_combined_poly and X_test_combined_poly
    # Reindex X_test_combined_poly to strictly match X_combined_poly's columns, filling missing with 0
    # This should now work without duplicate label errors if clean_feature_names is effective
    X_test_combined_poly = X_test_combined_poly.reindex(columns=X_combined_poly.columns, fill_value=0)

    print("Polynomial features created and combined with other features.")
    print(f"X_combined_poly shape: {X_combined_poly.shape}")
    print(f"X_test_combined_poly shape: {X_test_combined_poly.shape}")

    print(
"\n--- X_combined_poly Head ---")
    display(X_combined_poly.head())
    print("\n--- X_combined_poly Info ---")
    display(X_combined_poly.info())

    print("\n--- X_test_combined_poly Head ---")
    display(X_test_combined_poly.head())
    print("\n--- X_test_combined_poly Info ---")
    display(X_test_combined_poly.info())

    # Update X and X_test to the new combined dataframes for subsequent steps
    X = X_combined_poly
    X_test = X_test_combined_poly
else:
    print("X, X_test, or y not found. Please ensure previous data preparation steps were executed.")

### Prepare Data for Model Training (with Polynomial Features)

Now that polynomial features have been generated and combined, I will re-split the training data into training and validation sets using the enhanced feature set. This ensures the model is trained and evaluated on the new features.

In [ ]:
from sklearn.model_selection import train_test_split

if 'X' in locals() and 'y' in locals() and 'X_test' in locals():
    # X and X_test now contain polynomial features with cleaned names

    # Split the training data into training and validation sets
    X_train_poly, X_val_poly, y_train_poly, y_val_poly = train_test_split(X, y, test_size=0.2, random_state=42)

    print("Data split into training and validation sets for polynomial features.")
    print(f"X_train_poly shape: {X_train_poly.shape}, y_train_poly shape: {y_train_poly.shape}")
    print(f"X_val_poly shape: {X_val_poly.shape}, y_val_poly shape: {y_val_poly.shape}")
else:
    print("X, y, or X_test not found. Please ensure polynomial feature engineering was executed.")

Data split into training and validation sets for polynomial features.
X_train_poly shape: (61839, 247), y_train_poly shape: (61839,)
X_val_poly shape: (15460, 247), y_val_poly shape: (15460,)


### Hyperparameter Tuning with GridSearchCV (Polynomial Features)

I will re-run `GridSearchCV` to find the optimal hyperparameters for the LightGBM Regressor using the dataset enhanced with polynomial features. This will allow the model to leverage the new feature interactions effectively.

In [ ]:
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb

if 'X_train_poly' in locals() and 'y_train_poly' in locals():
    # Define the parameter grid to search
    param_grid_poly = {
        'n_estimators': [100, 200, 300],  # Number of boosting rounds
        'learning_rate': [0.01, 0.05, 0.1],  # Step size shrinkage
        'num_leaves': [20, 31, 40],  # Max number of leaves in one tree
        'max_depth': [5, 7, -1],  # Max tree depth, -1 means no limit
        'reg_alpha': [0.1, 0.5],  # L1 regularization
        'reg_lambda': [0.1, 0.5]  # L2 regularization
    }

    lgb_tuned_poly = lgb.LGBMRegressor(random_state=42)

    # Initialize GridSearchCV
    grid_search_poly = GridSearchCV(estimator=lgb_tuned_poly, param_grid=param_grid_poly,
                               cv=3, scoring='r2', verbose=1, n_jobs=-1)

    # Fit GridSearchCV to the training data with polynomial features (cleaned names)
    print("Starting GridSearchCV for hyperparameter tuning with polynomial features...")
    grid_search_poly.fit(X_train_poly, y_train_poly)

    print("GridSearchCV completed for polynomial features.")
    print(f"Best parameters found with polynomial features: {grid_search_poly.best_params_}")
    print(f"Best R2 score (cross-validated) with polynomial features: {grid_search_poly.best_score_:.4f}")

    # Update the model with the best parameters
    lgb_reg_poly_optimized = grid_search_poly.best_estimator_
    print("Optimized LightGBM Regressor model with polynomial features created.")
else:
    print("Training data (X_train_poly, y_train_poly) not found. Please ensure polynomial feature data preparation was executed.")

Starting GridSearchCV for hyperparameter tuning with polynomial features...
Fitting 3 folds for each of 324 candidates, totalling 972 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


ValueError: 
All the 972 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
972 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1398, in fit
    super().fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/engine.py", line 297, in train
    booster = Booster(params=params, train_set=train_set)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/pyth

### Evaluate Optimized Model (Polynomial Features)

I will now evaluate the performance of the LightGBM model that was optimized using polynomial features. This will provide an updated R2 score and RMSE, helping us assess the impact of the new features on model accuracy.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

if 'lgb_reg_poly_optimized' in locals() and 'X_val_poly' in locals() and 'y_val_poly' in locals():
    y_pred_val_poly = lgb_reg_poly_optimized.predict(X_val_poly)

    r2_poly = r2_score(y_val_poly, y_pred_val_poly)
    rmse_poly = np.sqrt(mean_squared_error(y_val_poly, y_pred_val_poly))

    print(f"Optimized Validation R2 Score with Polynomial Features: {r2_poly:.4f}")
    print(f"Optimized Validation RMSE with Polynomial Features: {rmse_poly:.4f}")

    if r2_poly >= 0.95:
        print("Achieved 95% R2 score on the validation set with polynomial features!")
    else:
        print("R2 score is still below 95% after polynomial features. Further improvements might be needed.")
else:
    print("Optimized polynomial model or validation data not found. Please ensure previous steps were executed.")

LightGBMError: The number of features in data (247) is not the same as it was in training data (67).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.

### Make Predictions on Test Data with Optimized Polynomial Model and Generate Submission File

Using the optimized LightGBM model trained with polynomial features, I will now generate predictions on the test dataset and create a new submission file named `submission_polynomial.csv`. This will reflect the final performance with the enhanced feature set.

In [ ]:
import pandas as pd
import numpy as np

if 'lgb_reg_poly_optimized' in locals() and 'X_test' in locals() and 'test_ids' in locals():
    # Ensure X_test has the same column names and order as X_train_poly used for training
    # This is crucial because X_test was updated globally to contain cleaned polynomial features
    # and also 'X_test' in global state corresponds to X_combined_poly after feature engineering

    y_pred_test_poly = lgb_reg_poly_optimized.predict(X_test)

    # The demand was log-transformed, so inverse transform the predictions
    y_pred_test_poly = np.expm1(y_pred_test_poly)

    # Ensure predictions are non-negative after inverse transformation
    y_pred_test_poly[y_pred_test_poly < 0] = 0

    submission_df_poly = pd.DataFrame({'Index': test_ids, 'demand': y_pred_test_poly})
    submission_df_poly.to_csv('submission_polynomial.csv', index=False) # New filename

    print("Predictions made on the test set with optimized polynomial features model.")
    print("Submission file 'submission_polynomial.csv' created successfully!")
    display(submission_df_poly.head())
else:
    print("Optimized polynomial model, test data, or test IDs not found. Please ensure previous steps were executed correctly.")

LightGBMError: The number of features in data (1507) is not the same as it was in training data (67).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.

### Model Training: XGBoost Regressor (New Model)

Given that the R2 score is still below target, I will now explore another robust gradient boosting algorithm, XGBoost. I'll train an initial XGBoost Regressor model and evaluate its baseline performance before moving to hyperparameter tuning.

In [ ]:
import xgboost as xgb

if 'X_train_poly' in locals() and 'y_train_poly' in locals():
    # Ensure all columns in X_train_poly are numerical (float) for XGBoost
    X_train_poly_processed = X_train_poly.astype(float)
    y_train_poly_processed = y_train_poly.astype(float)

    # Initialize XGBoost Regressor
    xgb_reg = xgb.XGBRegressor(random_state=42, n_jobs=-1) # Use all available cores

    # Train the model with polynomial features data, converting to NumPy arrays
    xgb_reg.fit(X_train_poly_processed.values, y_train_poly_processed.values)

    print("XGBoost Regressor model trained successfully with polynomial features.")
else:
    print("Training data (X_train_poly, y_train_poly) not found. Please ensure previous steps were executed.")

AttributeError: 'DataFrame' object has no attribute 'dtype'

### Model Evaluation: XGBoost Regressor (Baseline)

Evaluating the baseline performance of the newly trained XGBoost model on the validation set using polynomial features.

In [ ]:
if 'xgb_reg' in locals() and 'X_val_poly' in locals() and 'y_val_poly' in locals():
    # Ensure all columns in X_val_poly are numerical (float) for XGBoost prediction
    X_val_poly_processed = X_val_poly.astype(float)
    y_val_poly_processed = y_val_poly.astype(float)

    # Convert X_val_poly_processed to NumPy array for prediction
    y_pred_val_xgb = xgb_reg.predict(X_val_poly_processed.values)

    r2_xgb = r2_score(y_val_poly_processed, y_pred_val_xgb)
    rmse_xgb = np.sqrt(mean_squared_error(y_val_poly_processed, y_pred_val_xgb))

    print(f"XGBoost Validation R2 Score (Baseline): {r2_xgb:.4f}")
    print(f"XGBoost Validation RMSE (Baseline): {rmse_xgb:.4f}")

    if r2_xgb >= 0.95:
        print("Achieved 95% R2 score on the validation set with XGBoost baseline model!")
    else:
        print("XGBoost R2 score is below 95% (baseline). Hyperparameter tuning will be performed next.")
else:
    print("XGBoost model or validation data not found. Please ensure previous steps were executed.")

NotFittedError: need to call fit or load_model beforehand

### Hyperparameter Tuning with GridSearchCV (XGBoost)

To optimize the XGBoost model, I'll use `GridSearchCV` to find the best hyperparameters. This step is crucial for squeezing out maximum performance from the model with our enriched feature set.

In [ ]:
from sklearn.model_selection import GridSearchCV

if 'X_train_poly' in locals() and 'y_train_poly' in locals():
    # Ensure all columns in X_train_poly are numerical (float) for GridSearchCV
    X_train_poly_processed = X_train_poly.astype(float)
    y_train_poly_processed = y_train_poly.astype(float)

    # Define a smaller parameter grid for quicker tuning initially
    param_grid_xgb = {
        'n_estimators': [100, 200, 300], # Number of boosting rounds
        'learning_rate': [0.05, 0.1, 0.2], # Step size shrinkage
        'max_depth': [3, 5, 7], # Max tree depth
        'subsample': [0.7, 0.9], # Subsample ratio of the training instance
        'colsample_bytree': [0.7, 0.9] # Subsample ratio of columns when constructing each tree
    }

    xgb_tuned = xgb.XGBRegressor(random_state=42, n_jobs=-1)

    # Initialize GridSearchCV
    grid_search_xgb = GridSearchCV(estimator=xgb_tuned, param_grid=param_grid_xgb,
                               cv=3, scoring='r2', verbose=1, n_jobs=-1)

    # Fit GridSearchCV to the training data with polynomial features, converting to NumPy arrays
    print("Starting GridSearchCV for XGBoost hyperparameter tuning...")
    grid_search_xgb.fit(X_train_poly_processed.values, y_train_poly_processed.values)

    print("GridSearchCV completed for XGBoost.")
    print(f"Best parameters found for XGBoost: {grid_search_xgb.best_params_}")
    print(f"Best R2 score (cross-validated) for XGBoost: {grid_search_xgb.best_score_:.4f}")

    # Update the model with the best parameters
    xgb_reg_optimized = grid_search_xgb.best_estimator_
    print("Optimized XGBoost Regressor model created.")
else:
    print("Training data (X_train_poly, y_train_poly) not found. Please ensure previous steps were executed.")

Starting GridSearchCV for XGBoost hyperparameter tuning...
Fitting 3 folds for each of 108 candidates, totalling 324 fits


ValueError: 
All the 324 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
324 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/xgboost/sklearn.py", line 1343, in fit
    train_dmatrix, evals = _wrap_evaluation_matrices(
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/xgboost/sklearn.py", line 700, in _wrap_evaluation_matrices
    train_dmatrix = create_dmatrix(


### Evaluate Optimized XGBoost Model

Evaluating the performance of the hyperparameter-tuned XGBoost model on the validation set.

In [ ]:
if 'xgb_reg_optimized' in locals() and 'X_val_poly' in locals() and 'y_val_poly' in locals():
    # Ensure all columns in X_val_poly are numerical (float) for XGBoost prediction
    X_val_poly_processed = X_val_poly.astype(float)

    # Convert X_val_poly_processed to NumPy array for prediction
    y_pred_val_xgb_optimized = xgb_reg_optimized.predict(X_val_poly_processed.values)

    r2_xgb_optimized = r2_score(y_val_poly, y_pred_val_xgb_optimized)
    rmse_xgb_optimized = np.sqrt(mean_squared_error(y_val_poly, y_pred_val_xgb_optimized))

    print(f"Optimized XGBoost Validation R2 Score: {r2_xgb_optimized:.4f}")
    print(f"Optimized XGBoost Validation RMSE: {rmse_xgb_optimized:.4f}")

    if r2_xgb_optimized >= 0.95:
        print("Achieved 95% R2 score on the validation set with optimized XGBoost model!")
    else:
        print("Optimized XGBoost R2 score is still below 95%. Further exploration might be needed.")
else:
    print("Optimized XGBoost model or validation data not found. Please ensure previous steps were executed.")

Optimized XGBoost model or validation data not found. Please ensure previous steps were executed.


### Make Predictions on Test Data with Optimized XGBoost Model and Generate Submission File

Generating predictions on the test dataset using the optimized XGBoost model and creating a new submission file named `submission_xgboost.csv`.

In [ ]:
import pandas as pd
import numpy as np

if 'xgb_reg_optimized' in locals() and 'X_test' in locals() and 'test_ids' in locals():
    # Ensure all columns in X_test are numerical (float) for XGBoost prediction
    X_test_processed = X_test.astype(float)

    # Convert X_test_processed to NumPy array for prediction
    y_pred_test_xgb = xgb_reg_optimized.predict(X_test_processed.values)

    # The demand was log-transformed, so inverse transform the predictions
    y_pred_test_xgb = np.expm1(y_pred_test_xgb)

    # Ensure predictions are non-negative after inverse transformation
    y_pred_test_xgb[y_pred_test_xgb < 0] = 0

    submission_df_xgb = pd.DataFrame({'Index': test_ids, 'demand': y_pred_test_xgb})
    submission_df_xgb.to_csv('submission_xgboost.csv', index=False) # New filename

    print("Predictions made on the test set with optimized XGBoost model.")
    print("Submission file 'submission_xgboost.csv' created successfully!")
    display(submission_df_xgb.head())
else:
    print("Optimized XGBoost model, test data, or test IDs not found. Please ensure previous steps were executed correctly.")

Optimized XGBoost model, test data, or test IDs not found. Please ensure previous steps were executed correctly.


### Evaluate Optimized Model

Predicting on the validation set with the optimized model and recalculating R2 score and Mean Squared Error.

In [ ]:
if 'lgb_reg_optimized' in locals() and 'X_val' in locals() and 'y_val' in locals():
    y_pred_val_optimized = lgb_reg_optimized.predict(X_val)

    r2_optimized = r2_score(y_val, y_pred_val_optimized)
    rmse_optimized = np.sqrt(mean_squared_error(y_val, y_pred_val_optimized))

    print(f"Optimized Validation R2 Score: {r2_optimized:.4f}")
    print(f"Optimized Validation RMSE: {rmse_optimized:.4f}")

    if r2_optimized >= 0.95:
        print("Achieved 95% R2 score on the validation set with optimized model!")
    else:
        print("R2 score is still below 95% after optimization. Further tuning or feature engineering might be needed.")

    # Update the lgb_reg for final predictions to the optimized one
    lgb_reg = lgb_reg_optimized
    print("Current prediction model updated to optimized version.")
else:
    print("Optimized model or validation data not found. Please ensure previous steps were executed.")

### Make Predictions on Test Data with Optimized Model and Generate Submission File

Using the optimized LightGBM model to predict demand on the `X_test` dataset and creating a new `submission_optimized.csv` file.

In [ ]:
if 'lgb_reg' in locals() and 'X_test' in locals() and 'test_ids' in locals():
    y_pred_test_optimized = lgb_reg.predict(X_test)

    # Ensure predictions are non-negative
    y_pred_test_optimized[y_pred_test_optimized < 0] = 0

    submission_df_optimized = pd.DataFrame({'Index': test_ids, 'demand': y_pred_test_optimized})
    submission_df_optimized.to_csv('submission_optimized.csv', index=False)

    print("Predictions made on the test set with optimized model.")
    print("Submission file 'submission_optimized.csv' created successfully!")
    display(submission_df_optimized.head())
else:
    print("Optimized model, test data, or test IDs not found. Please ensure previous steps were executed correctly.")

### Prepare Data for Model Training

Separating features (X) and target (y) from the training data, and splitting it into training and validation sets. Ensuring column consistency between train and test sets.

In [ ]:
if not train_df.empty and not test_df.empty:
    # Define features (X) and target (y)
    X = train_df.drop(['Index', 'demand'], axis=1)
    y = train_df['demand']

    # Keep 'Index' for test_df for submission
    test_ids = test_df['Index']
    X_test = test_df.drop('Index', axis=1)

    # Align columns - very important after one-hot encoding
    train_cols = X.columns
    test_cols = X_test.columns

    missing_in_test = set(train_cols) - set(test_cols)
    for c in missing_in_test:
        X_test[c] = 0

    missing_in_train = set(test_cols) - set(train_cols)
    for c in missing_in_train:
        X[c] = 0

    X_test = X_test[train_cols] # Ensure the order of columns is the same

    print("Features and target separated.")
    print(f"X shape: {X.shape}, y shape: {y.shape}")
    print(f"X_test shape: {X_test.shape}")

    # Split the training data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")

Features and target separated.
X shape: (77299, 13), y shape: (77299,)
X_test shape: (41778, 13)
X_train shape: (61839, 13), y_train shape: (61839,)
X_val shape: (15460, 13), y_val shape: (15460,)


### Model Training: LightGBM Regressor

Initializing and training a LightGBM Regressor model on the `X_train` and `y_train` datasets.

In [ ]:
if 'X_train' in locals() and 'y_train' in locals():
    lgb_reg = lgb.LGBMRegressor(random_state=42)
    lgb_reg.fit(X_train, y_train)
    print("LightGBM Regressor model trained successfully.")
else:
    print("Training data (X_train, y_train) not found. Please ensure previous steps were executed.")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003020 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 561
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 13
[LightGBM] [Info] Start training from score 0.093784
LightGBM Regressor model trained successfully.


### Model Evaluation

Predicting on the validation set and calculating R2 score and Mean Squared Error to evaluate the model's performance.

In [ ]:
if 'lgb_reg' in locals() and 'X_val' in locals() and 'y_val' in locals():
    y_pred_val = lgb_reg.predict(X_val)

    r2 = r2_score(y_val, y_pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))

    print(f"Validation R2 Score: {r2:.4f}")
    print(f"Validation RMSE: {rmse:.4f}")

    if r2 >= 0.95:
        print("Achieved 95% R2 score on the validation set!")
    else:
        print("R2 score is below 95%. Further hyperparameter tuning or feature engineering might be needed.")
else:
    print("Model or validation data not found. Please ensure previous steps were executed.")

Validation R2 Score: 0.8493
Validation RMSE: 0.0552
R2 score is below 95%. Further hyperparameter tuning or feature engineering might be needed.


### Prepare Data for Model Training

Separating features (X) and target (y) from the training data, and splitting it into training and validation sets. Ensuring column consistency between train and test sets.

In [ ]:
if not train_df.empty and not test_df.empty:
    # Define features (X) and target (y)
    X = train_df.drop(['Index', 'demand'], axis=1)
    y = train_df['demand']

    # Keep 'Index' for test_df for submission
    test_ids = test_df['Index']
    X_test = test_df.drop('Index', axis=1)

    # Align columns - very important after one-hot encoding
    train_cols = X.columns
    test_cols = X_test.columns

    missing_in_test = set(train_cols) - set(test_cols)
    for c in missing_in_test:
        X_test[c] = 0

    missing_in_train = set(test_cols) - set(train_cols)
    for c in missing_in_train:
        X[c] = 0

    X_test = X_test[train_cols] # Ensure the order of columns is the same

    # Convert boolean columns to int to avoid LightGBM feature mismatch issues
    for col in X.select_dtypes(include='bool').columns:
        X[col] = X[col].astype(int)
    for col in X_test.select_dtypes(include='bool').columns:
        X_test[col] = X_test[col].astype(int)

    print("Features and target separated and boolean columns converted to integers.")
    print(f"X shape: {X.shape}, y shape: {y.shape}")
    print(f"X_test shape: {X_test.shape}")

    # Split the training data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # Convert boolean columns in X_train and X_val to int as well
    for col in X_train.select_dtypes(include='bool').columns:
        X_train[col] = X_train[col].astype(int)
    for col in X_val.select_dtypes(include='bool').columns:
        X_val[col] = X_val[col].astype(int)

    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")

Features and target separated and boolean columns converted to integers.
X shape: (77299, 13), y shape: (77299,)
X_test shape: (41778, 13)
X_train shape: (61839, 13), y_train shape: (61839,)
X_val shape: (15460, 13), y_val shape: (15460,)


### Model Training: LightGBM Regressor

Initializing and training a LightGBM Regressor model on the `X_train` and `y_train` datasets.

In [ ]:
if 'X_train' in locals() and 'y_train' in locals():
    lgb_reg = lgb.LGBMRegressor(random_state=42)
    lgb_reg.fit(X_train, y_train)
    print("LightGBM Regressor model trained successfully.")
else:
    print("Training data (X_train, y_train) not found. Please ensure previous steps were executed.")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018031 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 561
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 13
[LightGBM] [Info] Start training from score 0.093784
LightGBM Regressor model trained successfully.


### Model Evaluation

Predicting on the validation set and calculating R2 score and Mean Squared Error to evaluate the model's performance.

In [ ]:
if 'lgb_reg' in locals() and 'X_val' in locals() and 'y_val' in locals():
    y_pred_val = lgb_reg.predict(X_val)

    r2 = r2_score(y_val, y_pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))

    print(f"Validation R2 Score: {r2:.4f}")
    print(f"Validation RMSE: {rmse:.4f}")

    if r2 >= 0.95:
        print("Achieved 95% R2 score on the validation set!")
    else:
        print("R2 score is below 95%. Further hyperparameter tuning or feature engineering might be needed.")
else:
    print("Model or validation data not found. Please ensure previous steps were executed.")

Validation R2 Score: 0.8493
Validation RMSE: 0.0552
R2 score is below 95%. Further hyperparameter tuning or feature engineering might be needed.
